# Airline Satisfaction — Experiment 2: tree ensemble

**Runs entirely on Kaggle.** Three-fold stratified CV compares LightGBM, GPU XGBoost, and GPU CatBoost with fixed row-local service, travel-segment, and delay features. CPU work by LightGBM also runs on Kaggle. No laptop training, external labels, public prediction files, or pretrained predictions are used in this tree-model experiment.

Attach `playground-series-s6e10`, enable GPU T4 x2 and Internet, then **Save & Run All**. The script refuses to run outside Kaggle.

The 599,635-row training pool receives three-fold CV. Fold validation labels select early stopping, so CV scores are developmental. A separate 50,000-row selection set chooses among 15 predefined convex blends; another 50,000 report performance without selecting weights. These partitions are reused from E1, not a new independent test. All models are then refit on the full training data with median CV tree counts. GPU CatBoost is not bitwise deterministic.

Reference public AUC: E1 **0.95835**. Scores printed by this notebook are validation scores, not Kaggle leaderboard results. This notebook creates CSVs; it does not submit automatically.

Documentation: [XGBoost GPU](https://xgboost.readthedocs.io/en/stable/gpu/index.html), [CatBoost GPU](https://catboost.ai/docs/en/features/training-on-gpu), [LightGBM](https://lightgbm.readthedocs.io/).


In [ ]:
from pathlib import Path
import os
if not Path('/kaggle/working').is_dir(): raise RuntimeError('Run this experiment only on Kaggle.')
os.environ.update(OMP_NUM_THREADS='4',OPENBLAS_NUM_THREADS='4',MKL_NUM_THREADS='4')


In [ ]:
%pip install -q 'xgboost>=2.1,<3.1' 'catboost>=1.2,<1.3'


In [ ]:
import torch,platform
if not torch.cuda.is_available(): raise RuntimeError('Enable GPU in Kaggle Settings before running.')
print('Execution: Kaggle',platform.node())
print('GPU:',torch.cuda.get_device_name(0),'count:',torch.cuda.device_count())
paths=[p.parent for p in Path('/kaggle/input').rglob('train.csv') if 'playground-series-s6e10' in str(p) and (p.parent/'test.csv').exists()]
if len(paths)!=1: raise ValueError(f'Attach exactly one copy of the competition: {paths}')
DATA_DIR=paths[0]
print('Competition data:',DATA_DIR)


In [ ]:
source='"""Kaggle-only E2: three tree families, three-fold CV, fixed features and blends."""\nfrom pathlib import Path\nimport os,sys,json,time,hashlib,importlib.metadata,gc\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import StratifiedKFold,train_test_split\nfrom sklearn.metrics import roc_auc_score,log_loss\n\nNAMES=[\'lightgbm\',\'xgboost\',\'catboost\']\n\ndef sha(p):\n    h=hashlib.sha256()\n    with open(p,\'rb\') as f:\n        for b in iter(lambda:f.read(1048576),b\'\'):h.update(b)\n    return h.hexdigest()\n\ndef feature_frame(d):\n    x=d.drop(columns=[\'id\',\'satisfaction\'],errors=\'ignore\').copy()\n    ratings=[\'Inflight wifi service\',\'Departure/Arrival time convenient\',\'Ease of Online booking\',\'Gate location\',\'Food and drink\',\'Online boarding\',\'Seat comfort\',\'Inflight entertainment\',\'On-board service\',\'Leg room service\',\'Baggage handling\',\'Checkin service\',\'Cleanliness\']\n    r=x[ratings]\n    x[\'rating_mean\']=r.mean(axis=1);x[\'rating_std\']=r.std(axis=1)\n    x[\'rating_min\']=r.min(axis=1);x[\'rating_max\']=r.max(axis=1)\n    x[\'rating_low_count\']=(r<=2).sum(axis=1);x[\'rating_high_count\']=(r>=4).sum(axis=1);x[\'rating_zero_count\']=(r==0).sum(axis=1)\n    x[\'digital_mean\']=x[[\'Inflight wifi service\',\'Ease of Online booking\',\'Online boarding\']].mean(axis=1)\n    x[\'cabin_mean\']=x[[\'Seat comfort\',\'Inflight entertainment\',\'Leg room service\',\'Cleanliness\']].mean(axis=1)\n    x[\'wifi_boarding_gap\']=x[\'Inflight wifi service\']-x[\'Online boarding\']\n    x[\'seat_entertainment_gap\']=x[\'Seat comfort\']-x[\'Inflight entertainment\']\n    dep=x[\'Departure Delay in Minutes\'];arr=x[\'Arrival Delay in Minutes\']\n    x[\'arrival_missing\']=arr.isna().astype(\'int8\');x[\'delay_change\']=arr-dep\n    x[\'departure_log\']=np.log1p(dep.clip(lower=0));x[\'arrival_log\']=np.log1p(arr.clip(lower=0))\n    x[\'distance_log\']=np.log1p(x[\'Flight Distance\'])\n    x[\'travel_segment\']=x[\'Class\'].astype(str)+\'|\'+x[\'Type of Travel\'].astype(str)+\'|\'+x[\'Customer Type\'].astype(str)\n    return x\n\ndef encode(a,others,name):\n    a=a.copy();others=[b.copy() for b in others]\n    cats=list(a.select_dtypes(include=[\'object\',\'string\']).columns)\n    if name==\'catboost\':\n        for c in cats:\n            a[c]=a[c].fillna(\'Missing\').astype(str)\n            for b in others:b[c]=b[c].fillna(\'Missing\').astype(str)\n    else:\n        for c in cats:\n            levels=sorted(a[c].dropna().unique().tolist())\n            a[c]=pd.Categorical(a[c],categories=levels)\n            for b in others:b[c]=pd.Categorical(b[c],categories=levels)\n    # XGBoost prohibits these characters in feature names; original names currently have none.\n    safe={c:c.replace(\'[\',\'_\').replace(\']\',\'_\').replace(\'<\',\'_\') for c in a.columns}\n    a=a.rename(columns=safe);others=[b.rename(columns=safe) for b in others]\n    return a,others,[safe[c] for c in cats]\n\ndef make_model(name,trees=3000,early=False,seed=42):\n    if name==\'lightgbm\':\n        import lightgbm as lgb\n        return lgb.LGBMClassifier(n_estimators=trees,objective=\'binary\',learning_rate=.035,\n            num_leaves=63,max_depth=-1,min_child_samples=120,colsample_bytree=.85,\n            subsample=.8,subsample_freq=1,reg_lambda=8,n_jobs=4,random_state=seed,\n            verbosity=-1,deterministic=True,force_col_wise=True)\n    if name==\'xgboost\':\n        from xgboost import XGBClassifier\n        return XGBClassifier(n_estimators=trees,objective=\'binary:logistic\',eval_metric=\'auc\',\n            learning_rate=.035,max_depth=6,min_child_weight=25,subsample=.85,\n            colsample_bytree=.85,reg_lambda=12,reg_alpha=.1,max_bin=256,\n            tree_method=\'hist\',device=\'cuda\',enable_categorical=True,n_jobs=4,\n            random_state=seed,**({\'early_stopping_rounds\':120} if early else {}))\n    from catboost import CatBoostClassifier\n    return CatBoostClassifier(iterations=trees,depth=7,learning_rate=.05,l2_leaf_reg=8,\n        loss_function=\'Logloss\',eval_metric=\'AUC\',task_type=\'GPU\',devices=\'0\',\n        random_seed=seed,thread_count=4,allow_writing_files=False,verbose=300,\n        **({\'od_type\':\'Iter\',\'od_wait\':120} if early else {}))\n\ndef metrics(y,p):return {\'auc\':float(roc_auc_score(y,p)),\'log_loss\':float(log_loss(y,p))}\n\ndef run(data_dir,out):\n    if not Path(\'/kaggle/working\').is_dir():raise RuntimeError(\'This experiment must run on Kaggle, never on the laptop.\')\n    import torch\n    if not torch.cuda.is_available():raise RuntimeError(\'Enable Kaggle GPU accelerator.\')\n    print(\'HOST: Kaggle; GPU:\',torch.cuda.get_device_name(0),\'device count:\',torch.cuda.device_count(),flush=True)\n    start=time.time();out=Path(out);out.mkdir(parents=True,exist_ok=True);data_dir=Path(data_dir)\n    tr=pd.read_csv(data_dir/\'train.csv\');te=pd.read_csv(data_dir/\'test.csv\');sample=pd.read_csv(data_dir/\'sample_submission.csv\')\n    y=tr.satisfaction.astype(int).to_numpy();x=feature_frame(tr);xt=feature_frame(te)\n    assert tr.id.is_unique and te.id.is_unique and np.array_equal(te.id,sample.id)\n    assert list(x)==list(xt) and not set(tr.id)&set(te.id)\n    # Preserve E1\'s partitions. This is reused housing-free airline data, not a new independent dataset.\n    pool,held=train_test_split(np.arange(len(y)),test_size=100000,random_state=42,stratify=y)\n    selection,report=train_test_split(held,test_size=.5,random_state=2026,stratify=y[held])\n    mask=np.isin(held,selection);assert not set(pool)&set(held)\n    versions={p:importlib.metadata.version(p) for p in [\'lightgbm\',\'xgboost\',\'catboost\',\'pandas\',\'numpy\',\'scikit-learn\',\'torch\']}\n    config=dict(source_sha256=sha(__file__),train_sha256=sha(data_dir/\'train.csv\'),test_sha256=sha(data_dir/\'test.csv\'),versions=versions,\n                folds=3,seed=20261028,feature_columns=list(x),models=NAMES,blend_grid_step=.25,\n                host=\'Kaggle\',gpu=torch.cuda.get_device_name(0),baseline_public_auc=.95835)\n    fp=hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest();cache=out/f\'cache_{fp[:16]}\';cache.mkdir(exist_ok=True)\n    (out/\'config.json\').write_text(json.dumps(config,indent=2))\n    print(\'DATA:\',tr.shape,te.shape,\'features:\',x.shape[1],\'training pool:\',len(pool),\'held selection/report:\',len(selection),len(report),flush=True)\n    folds=list(StratifiedKFold(3,shuffle=True,random_state=20261028).split(pool,y[pool]))\n    oof=np.zeros((len(pool),3));hp=np.zeros((len(held),3));fold_records=[];best={n:[] for n in NAMES}\n    for fold,(a,b) in enumerate(folds):\n        ti,vi=pool[a],pool[b]\n        for j,name in enumerate(NAMES):\n            f=cache/f\'{name}_fold{fold}.npz\'\n            print(\'START\',name,\'fold\',fold+1,flush=True)\n            if f.exists():\n                z=np.load(f);pv=z[\'validation\'];ph=z[\'held\'];iterations=int(z[\'iterations\'])\n                assert np.array_equal(z[\'validation_ids\'],tr.id.iloc[vi])\n            else:\n                aa,bb,cats=encode(x.iloc[ti],[x.iloc[vi],x.iloc[held]],name);av,ah=bb\n                model=make_model(name,early=True,seed=42+fold)\n                if name==\'lightgbm\':\n                    import lightgbm as lgb\n                    model.fit(aa,y[ti],eval_set=[(av,y[vi])],eval_metric=\'auc\',callbacks=[lgb.early_stopping(120,verbose=False),lgb.log_evaluation(300)])\n                    iterations=int(model.best_iteration_)\n                elif name==\'xgboost\':\n                    model.fit(aa,y[ti],eval_set=[(av,y[vi])],verbose=300);iterations=int(model.best_iteration)+1\n                    if model.get_booster().save_config().find(\'cuda\')<0:raise RuntimeError(\'XGBoost did not use GPU\')\n                else:\n                    model.fit(aa,y[ti],cat_features=cats,eval_set=(av,y[vi]),use_best_model=True);iterations=int(model.tree_count_)\n                pv=model.predict_proba(av)[:,1];ph=model.predict_proba(ah)[:,1]\n                np.savez_compressed(f,validation=pv,held=ph,iterations=iterations,validation_ids=tr.id.iloc[vi].to_numpy())\n                del model,aa,av,ah,bb;gc.collect();torch.cuda.empty_cache()\n            assert np.isfinite(pv).all() and np.isfinite(ph).all()\n            oof[b,j]=pv;hp[:,j]+=ph/3;best[name].append(iterations)\n            record=dict(model=name,fold=fold+1,iterations=iterations,**metrics(y[vi],pv));fold_records.append(record)\n            print(\'DONE\',record,\'elapsed seconds:\',round(time.time()-start),flush=True)\n            (out/\'fold_metrics.json\').write_text(json.dumps(fold_records,indent=2))\n    # Small fixed grid selected on selection holdout only; report labels do not choose weights.\n    weights=[np.array([a,b,4-a-b])/4 for a in range(5) for b in range(5-a)]\n    candidates=[{\'weights\':w.tolist(),**metrics(y[held][mask],(hp@w)[mask])} for w in weights]\n    win=max(candidates,key=lambda c:c[\'auc\']);w=np.asarray(win[\'weights\'])\n    result=dict(config=config,fingerprint=fp,folds=fold_records,\n        cv_selection_note=\'Fold labels choose early stopping, so CV is developmental. Blends selected on separate 50k selection holdout. Report holdout reused from E1, not fresh independent data.\',\n        cv={name:metrics(y[pool],oof[:,j]) for j,name in enumerate(NAMES)},selection_candidates=candidates,\n        selected_weights=dict(zip(NAMES,w.tolist())),report={name:metrics(y[held][~mask],hp[~mask,j]) for j,name in enumerate(NAMES)})\n    result[\'report\'][\'selected\']=metrics(y[held][~mask],(hp@w)[~mask]);(out/\'metrics.json\').write_text(json.dumps(result,indent=2))\n    print(\'REPORT\',json.dumps(result[\'report\']),\'SELECTED WEIGHTS\',result[\'selected_weights\'],flush=True)\n    pd.DataFrame({\'id\':tr.id.iloc[pool].to_numpy(),\'y\':y[pool],**{n:oof[:,j] for j,n in enumerate(NAMES)}}).to_csv(out/\'oof_predictions.csv\',index=False)\n    pd.DataFrame({\'id\':tr.id.iloc[held].to_numpy(),\'y\':y[held],\'partition\':np.where(mask,\'selection\',\'report\'),**{n:hp[:,j] for j,n in enumerate(NAMES)}}).to_csv(out/\'held_predictions.csv\',index=False)\n    tp=np.zeros((len(te),3))\n    for j,name in enumerate(NAMES):\n        f=cache/f\'{name}_full.npy\';iterations=int(np.median(best[name]));print(\'FULL REFIT\',name,iterations,flush=True)\n        if f.exists():tp[:,j]=np.load(f)\n        else:\n            aa,bb,cats=encode(x,[xt],name);at=bb[0];model=make_model(name,trees=iterations)\n            if name==\'catboost\':model.fit(aa,y,cat_features=cats);model.save_model(str(out/\'catboost.cbm\'))\n            else:\n                model.fit(aa,y)\n                if name==\'xgboost\':model.save_model(out/\'xgboost.ubj\')\n                else:model.booster_.save_model(str(out/\'lightgbm.txt\'))\n            tp[:,j]=model.predict_proba(at)[:,1];np.save(f,tp[:,j])\n            (out/f\'{name}_preprocessing.json\').write_text(json.dumps({\'features\':list(aa),\'categories\':{c:list(aa[c].cat.categories) for c in aa.select_dtypes(\'category\')}},indent=2))\n            del model,aa,at,bb;gc.collect();torch.cuda.empty_cache()\n        assert np.isfinite(tp[:,j]).all() and ((tp[:,j]>=0)&(tp[:,j]<=1)).all()\n        s=sample.copy();s.satisfaction=tp[:,j];s.to_csv(out/f\'submission_{name}.csv\',index=False)\n    s=sample.copy();s.satisfaction=tp@w;assert s.satisfaction.between(0,1).all();s.to_csv(out/\'submission_selected.csv\',index=False)\n    result.update(runtime_seconds=time.time()-start,submission_sha256=sha(out/\'submission_selected.csv\'))\n    (out/\'metrics.json\').write_text(json.dumps(result,indent=2));print(\'COMPLETED\',result[\'runtime_seconds\'],flush=True)\n    return result\n\nif __name__==\'__main__\':\n    import argparse\n    p=argparse.ArgumentParser();p.add_argument(\'--data-dir\',required=True);p.add_argument(\'--output-dir\',default=\'/kaggle/working/experiment2\');a=p.parse_args();run(a.data_dir,a.output_dir)\n'
Path('/kaggle/working/run_experiment2.py').write_text(source)


In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-u','/kaggle/working/run_experiment2.py','--data-dir',str(DATA_DIR)],check=True)


In [ ]:
import json,pandas as pd
from IPython.display import display,FileLink
metrics=json.loads(Path('/kaggle/working/experiment2/metrics.json').read_text())
display(pd.DataFrame(metrics['cv']).T)
display(pd.DataFrame(metrics['report']).T)
print('Selected blend:',metrics['selected_weights'])
display(FileLink('/kaggle/working/experiment2/submission_selected.csv'))
